# Lenguajes y Autómatas II (SCD-1016)
## Sesión 40 · Hito 4: Demostración Final del Compilador End-to-End

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Puede el compilador que construiste llevar un programa del texto fuente hasta un resultado correcto sin intervención manual?
>
> **Objetivo de la sesión:** Cerrar el pipeline ejecutando un mini lenguaje de asignaciones y return de principio a fin, y demostrar que las decisiones de las nueve sesiones anteriores encajan en un solo flujo.

**Entregable:** tu implementación de ejecutar_programa() en artesano_sesion40_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion40.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes un programa del mini lenguaje separado por punto y coma, con asignaciones x = expr y una sentencia return expr, y devuelves el valor final; cada expresión usa a lo sumo un operador entre dos átomos. El harness prueba seis programas representativos, mientras que la batería completa de diez vive en el proyecto del Hito 4.

Al final de la sesión tendrás: **Una función ejecutar_programa que interpreta asignaciones y return de un mini lenguaje, integrando todas las fases del compilador en una sola llamada.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 40](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion40.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion40.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion40.h"

long ejecutar_programa(const char *fuente);
```

- Las variables no asignadas valen cero y conservan su valor hasta el final del programa.
- Solo se admiten los operadores +, - y *, aplicados una vez por expresión.
- Si no hay return o el programa está vacío, el resultado es cero; NULL también devuelve cero.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion40_c.c` junto con `artesano_sesion40_c.c`
> (o `ia_sesion40_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `x = 5; return x + 3` | 8 |
| 2 | `x = 2; y = x * 3; return y - 1` | 5 |
| 3 | `a = 4; b = a * a; return b` | 16 |
| 4 | `return 7` | 7 |
| 5 | `x = 0; return x` | 0 |
| 6 | `return z` | 0 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 40 - Compilador Modular Funcional Completo ===
Caso                               esperado:              obtenido:              resultado
[ 1] x = 5; return x + 3                esperado: 8            obtenido: 8            OK
[ 2] x = 2; y = x * 3; return y - 1     esperado: 5            obtenido: 5            OK
[ 3] a = 4; b = a * a; return b         esperado: 16           obtenido: 16           OK
[ 4] return 7                           esperado: 7            obtenido: 7            OK
[ 5] x = 0; return x                    esperado: 0            obtenido: 0            OK
[ 6] return z                           esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion40.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] ejecutar_programa respeta el orden de las asignaciones y resuelve variables ya definidas.
- [ ] El resultado coincide con el oráculo en programas con una o varias sentencias, incluidos los casos vacíos.
- [ ] Los seis casos del harness salen OK y el programa termina con código de salida 0.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion40_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion40.h
/* api_sesion40.h - contrato de la Sesion 40
 * Compilador Modular Funcional Completo
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION40_H
#define API_SESION40_H

long ejecutar_programa(const char *fuente);

#endif /* API_SESION40_H */


In [ ]:
%%writefile artesano_sesion40_c.c
/* artesano_sesion40_c.c - Fase 1: El Artesano (sin IA)
 * Compilador Modular Funcional Completo.
 *
 * Implementa lo que declara api_sesion40.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion40.h"
#include <stdio.h>
#include <string.h>

long ejecutar_programa(const char *fuente) {
    (void)fuente; /* TODO: recorrer sentencias, guardar variables y resolver return */
    return 0;
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion40_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion40_c.c
/* ia_sesion40_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion40.h. No puede traer main().
 */
#include "api_sesion40.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion40_c.c
/* harness_sesion40_c.c - auditoria automatica de la Sesion 40
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion40_c.c artesano_sesion40_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion40.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 40 - Compilador Modular Funcional Completo ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { long r = ejecutar_programa("x = 5; return x + 3"); verificar_int("x = 5; return x + 3", (long)(8), (long)(r)); }
    { long r = ejecutar_programa("x = 2; y = x * 3; return y - 1"); verificar_int("x = 2; y = x * 3; return y - 1", (long)(5), (long)(r)); }
    { long r = ejecutar_programa("a = 4; b = a * a; return b"); verificar_int("a = 4; b = a * a; return b", (long)(16), (long)(r)); }
    { long r = ejecutar_programa("return 7"); verificar_int("return 7", (long)(7), (long)(r)); }
    { long r = ejecutar_programa("x = 0; return x"); verificar_int("x = 0; return x", (long)(0), (long)(r)); }
    { long r = ejecutar_programa("return z"); verificar_int("return z", (long)(0), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion40.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion40_c.c artesano_sesion40_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion40_c.c ia_sesion40_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion40.md` a `auditorias/auditoria_sesion40.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
